## Task 9: Real-Time Vector Stream Ingestion & Spark Reindexing
**Requires:** a running Apache Kafka broker, PySpark, and a Qdrant vector database. Not available in this sandbox — reference implementation below.

In [1]:
!apt-get install -y openjdk-17-jdk-headless -qq
!wget https://archive.apache.org/dist/kafka/3.7.0/kafka_2.13-3.7.0.tgz
!tar -xzf kafka_2.13-3.7.0.tgz
!ls

--2026-08-09 05:33:02--  https://archive.apache.org/dist/kafka/3.7.0/kafka_2.13-3.7.0.tgz
Resolving archive.apache.org (archive.apache.org)... 65.108.204.189, 2a01:4f9:1a:a084::2
Connecting to archive.apache.org (archive.apache.org)|65.108.204.189|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 119028138 (114M) [application/x-gzip]
Saving to: ‘kafka_2.13-3.7.0.tgz’

kafka_2.13-3.7.0.tg 100%[===================>] 113.51M  13.3MB/s    in 18s     

2026-08-09 05:33:20 (6.49 MB/s) - ‘kafka_2.13-3.7.0.tgz’ saved [119028138/119028138]

kafka_2.13-3.7.0  kafka_2.13-3.7.0.tgz	sample_data


In [2]:
%cd kafka_2.13-3.7.0
!rm -rf /tmp/kraft-combined-logs
!bin/kafka-storage.sh random-uuid > /tmp/cluster_id.txt
!bin/kafka-storage.sh format -t $(cat /tmp/cluster_id.txt) -c config/kraft/server.properties

import subprocess, time
subprocess.Popen(['bin/kafka-server-start.sh', 'config/kraft/server.properties'])
time.sleep(15)
print("Kafka broker starting...")

/content/kafka_2.13-3.7.0
metaPropertiesEnsemble=MetaPropertiesEnsemble(metadataLogDir=Optional.empty, dirs={/tmp/kraft-combined-logs: EMPTY})
Formatting /tmp/kraft-combined-logs with metadata.version 3.7-IV4.
Kafka broker starting...


In [3]:
!bin/kafka-topics.sh --create --topic transaction-logs --bootstrap-server localhost:9092

Created topic transaction-logs.


In [4]:
!(echo "customer bought a laptop"; echo "customer returned a phone"; echo "customer bought headphones") | bin/kafka-console-producer.sh --topic transaction-logs --bootstrap-server localhost:9092

In [5]:
!pip uninstall pyspark -y -q
!pip install pyspark==3.5.0 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 22.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.0 which is incompatible.


In [7]:
!pip install qdrant-client -q

import os

# Explicitly unset PYSPARK_SUBMIT_ARGS to prevent conflicts with SparkSession.builder
# This must be done *before* importing pyspark to be effective.
if 'PYSPARK_SUBMIT_ARGS' in os.environ:
    del os.environ['PYSPARK_SUBMIT_ARGS']

# Now import pyspark and other libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, VectorParams, Distance

# Stop any existing SparkSession to ensure the new configuration is picked up
if 'spark' in locals() and spark is not None:
    spark.stop()

spark = SparkSession.builder \
    .appName("VectorStream") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,org.apache.kafka:kafka-clients:3.5.1") \
    .getOrCreate()

model = SentenceTransformer('all-MiniLM-L6-v2')

# in-memory Qdrant — the real engine, running inside this process instead of a separate server
qdrant = QdrantClient(location=":memory:")
qdrant.create_collection(
    collection_name="live_stream",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

stream_df = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "transaction-logs")
    .option("startingOffsets", "earliest")
    .load())

point_id_counter = [0]

def process_batch(batch_df, batch_id):
    rows = batch_df.select(col("value").cast("string")).collect()
    texts = [r["value"] for r in rows]
    if not texts:
        return
    vectors = model.encode(texts)
    points = []
    for v, t in zip(vectors, texts):
        points.append(PointStruct(id=point_id_counter[0], vector=v.tolist(), payload={"text": t}))
        point_id_counter[0] += 1
    qdrant.upsert(collection_name="live_stream", points=points)
    print(f"Batch {batch_id}: ingested {len(points)} messages -> Qdrant")

query = stream_df.writeStream.foreachBatch(process_batch).start()
query.awaitTermination(timeout=30)   # run for 30s instead of forever, so the cell actually finishes
query.stop()

# prove it worked: pull back what's actually in the vector DB now
results = qdrant.scroll(collection_name="live_stream", limit=10)
print("\nStored in Qdrant:")
for point in results[0]:
    print(f"  id={point.id}  text={point.payload['text']}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batch 0: ingested 3 messages -> Qdrant

Stored in Qdrant:
  id=0  text=customer bought a laptop
  id=1  text=customer returned a phone
  id=2  text=customer bought headphones
